In [1]:
import os
import copy
import json
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random as random
import numpy as np
from ray import tune
import numpy as np
from sklearn import metrics

In [2]:
    neg_to_pos_ratio = 1

    results_file = "results_titles_abstracts_gbdt.json"

    inputs = "titles_abstracts"

    filename_pos_human = "../data/pubmed_articles_pos_human.csv"
    filename_neg_human = "../data/pubmed_articles_neg_human.csv"

    with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
        prop_typ_id_dict = pkl.load(f)
    ptype = 'is_sr'

In [4]:
all_ids = []
for ptype in prop_typ_id_dict.keys():
    for pos_neg in ["pos", "neg"]:
        for split in ["train", "val", "test"]:
            all_ids.extend(prop_typ_id_dict[ptype][pos_neg][split])

In [7]:
all_ids = list(set(all_ids))

In [8]:
len(all_ids)

118709

In [9]:
pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)

In [11]:
pos_set = pd.DataFrame()
for pos_chunk in tqdm(pos_data_human):

                indices = pos_chunk.index
                indices = list(indices)

                pos_indices = list(set(indices).intersection(set(all_ids)))

                pos_subset = pos_chunk.loc[pos_indices]
                if pos_set.empty:
                    pos_set = pd.DataFrame(columns=pos_set.columns)
                pos_set = pd.concat([pos_set, pos_subset])

18it [00:24,  1.42s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
50it [01:09,  1.44s/it]/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
85it [01:57,  1.38s/it]


In [12]:
neg_set = pd.DataFrame()
for neg_chunk in tqdm(neg_data_human):

                indices = neg_chunk.index
                indices = list(indices)

                neg_indices = list(set(indices).intersection(set(all_ids)))

                neg_subset = neg_chunk.loc[neg_indices]
                if neg_set.empty:
                    neg_set = pd.DataFrame(columns=neg_set.columns)
                neg_set = pd.concat([neg_set, neg_subset])

57it [01:27,  1.54s/it]


In [16]:
all_data = pd.concat([neg_set, pos_set]).drop_duplicates()

In [20]:
all_data.ptyp

65540                          ['Journal Article', 'Review']
24587      ['Journal Article', "Research Support, Non-U.S...
73740                                    ['Journal Article']
17                                       ['Journal Article']
8209                                     ['Journal Article']
                                 ...                        
8509421    ['English Abstract', 'Journal Article', 'Rando...
8542196                        ['Journal Article', 'Review']
8583163    ['Case Reports', 'Journal Article', 'Review', ...
8581117    ['Comparative Study', 'Journal Article', "Rese...
8542207           ['Journal Article', 'Observational Study']
Name: ptyp, Length: 215685, dtype: object

In [25]:
num_multilabel = 0
for ptype in all_data.ptyp:
    if len(eval(ptype)) > 1:
        num_multilabel +=1
print(f"proportion multilabel: {num_multilabel/len(all_data)}")

proportion multilabel: 0.7961286134872615


In [28]:
all_data.to_pickle("all_data.pkl")